# ***📌Portfolio Project 2***



***Мета проекту*** - проаналізувати результати A/B-тестування за допомогою статистичних методів в Python та створити візуалізацію, що демонструє ключові конверсійні метрики.


**Етап 1. Розрахунок статистичної значущості.**

Розрахунок статистичної значущісті для чотирьох метрик:

* add_payment_info / session

* add_shipping_info / session

* begin_checkout / session

* new_accounts / session

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from google.colab import drive, files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
file_path = '/content/drive/My Drive/AB_tool.csv'
df = pd.read_csv(file_path)
df

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new_account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new_account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new_account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new_account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new_account,1
...,...,...,...,...,...,...,...,...,...
800991,2020-12-19,Vietnam,mobile,Asia,Direct,3,2,session,1
800992,2020-12-19,Vietnam,mobile,Asia,Undefined,3,2,session,1
800993,2020-12-19,Vietnam,mobile,Asia,Organic Search,3,2,session,1
800994,2020-12-19,Vietnam,desktop,Asia,Organic Search,3,2,session,1


In [3]:
df.columns

Index(['date', 'country', 'device', 'continent', 'channel', 'test',
       'test_group', 'event_name', 'value'],
      dtype='object')

In [4]:
df['test_group'].unique()

array([2, 1])

In [5]:
df = df.rename(columns={'test': 'test_number',
                        'test_group': 'variant'})
df.head()

,date,country,device,continent,channel,test_number,variant,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new_account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new_account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new_account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new_account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new_account,1


In [6]:
df['variant'] = df['variant'].map({1: 'control', 2: 'test'})
df.head()

,date,country,device,continent,channel,test_number,variant,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,test,new_account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,test,new_account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,test,new_account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,control,new_account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,test,new_account,1


In [7]:
df['event_name'].unique()

array(['new_account', 'session with orders', 'user_engagement',
       'first_visit', 'page_view', 'scroll', 'session_start', 'view_item',
       'view_promotion', 'view_search_results', 'select_item',
       'add_to_cart', 'begin_checkout', 'add_shipping_info',
       'select_promotion', 'add_payment_info', 'click', 'session',
       'view_item_list'], dtype=object)

In [8]:
events = ['add_payment_info', 'add_shipping_info', 'begin_checkout',  'new_account', 'session' ]

df = df[df['event_name'].isin(events)]
df['event_name'].unique()

array(['new_account', 'begin_checkout', 'add_shipping_info',
       'add_payment_info', 'session'], dtype=object)

In [9]:
df = df.pivot_table(index=['date', 'country', 'device', 'continent', 'channel', 'test_number', 'variant'],
                    columns='event_name', values='value', aggfunc='sum').reset_index()

df.columns.name = None

In [10]:
for col in events:
  if col not in df.columns:
    df[col] = 0

df = df.fillna(0)

print('DataSet Shape:', df.shape)

DataSet Shape: (107210, 12)


In [11]:
df.head()

,date,country,device,continent,channel,test_number,variant,add_payment_info,add_shipping_info,begin_checkout,new_account,session
0,2020-11-01,(not set),desktop,Africa,Paid Search,1,control,0.0,0.0,0.0,0.0,1.0
1,2020-11-01,(not set),desktop,Africa,Paid Search,2,control,0.0,0.0,0.0,0.0,1.0
2,2020-11-01,(not set),desktop,Americas,Direct,1,test,0.0,0.0,0.0,1.0,1.0
3,2020-11-01,(not set),desktop,Americas,Direct,2,control,0.0,0.0,0.0,1.0,1.0
4,2020-11-01,(not set),desktop,Americas,Paid Search,1,control,0.0,1.0,1.0,0.0,2.0


events = ['add_payment_info', 'add_shipping_info', 'begin_checkout',  'new_account', 'session' ]


In [12]:
metric_name = [('add_payment_info', 'session'),
           ('add_shipping_info', 'session'),
           ('begin_checkout', 'session'),
           ('new_account', 'session')]

***Порівняння пропорцій (control vs test)***


- **num_control**: кількість разів скільки подія відбулась у control;
- **den_control**: всього сесій у control;
- **num_test**: кількість разів скільки подія відбулась у test;
- **den_test**: всього сесій у test;
- **conv1, conv2**: конверсія відповідної групи;
- **p_pool**: середня пропорція по обох групах разом, для визначення оцінки розкиду даних за умови, що нульова гіпотеза правдива:
- **se**: sqrt error - помилка, наскільки сильно коливається різниця між групами через випадковість;
- **z_stat**: на скільки стандартних помилок test відрізняється від control;
- **p_value < 0.05**: True (результат значущий, зміна реальна), False (різниця могла виникнути випадково)

In [13]:
def z_test_proportion(num_control, den_control, num_test, den_test):
  conv1 = num_control / den_control if den_control > 0 else 0
  conv2 = num_test / den_test if den_test > 0 else 0

  p_pool = (num_control +num_test) / (den_control + den_test)
  se = np.sqrt(p_pool * (1 - p_pool) * (1 / den_control + 1 / den_test))

  z_stat  = (conv2 - conv1) / se if se > 0 else 0
  p_value = 1 - stats.norm.cdf(z_stat)

  return conv1, conv2, conv2 - conv1, z_stat, p_value, p_value < 0.05

# ***✅Аналіз по тесту (тотал)***

In [14]:
def cal_signif_total(data):
  group_cols = ['test_number']
  metric_cols = list({col for col, _ in metric_name } | {'session'})

# 1. Агрегація
# Просумувати всі події по групах (test_number + variant)
# Результат -  скільки разів кожна подія відбулась у control і test.
  agg = (data.groupby(group_cols + ['variant'])[metric_cols].sum().reset_index())

# 2. Розділення груп. Розділяємо зведену таблицю на дві окремі.
  control = agg[agg['variant'] == 'control'].drop(columns='variant')
  test = agg[agg['variant'] == 'test'].drop(columns='variant')

# 3. Об'єднання. Об'єднання control і test в один рядок по test_number
# Суфікси _control і _test розрізняють колонки двох груп.
  merged = control.merge(test, on=group_cols, suffixes=('_control', '_test'))

  records = []
# 4. Цикл по метрикам. Для кожної метрики з metric_name динамічно:
# 1. Беремо числівник і знаменник для control і test
# 2. Рахуємо z-test для кожного рядка
# 3. Збираємо результат у датафрейм.

  for numerator_col, denominator_col in metric_name:
        num_c = merged[f'{numerator_col}_control']
        den_c = merged[f'{denominator_col}_control']
        num_e = merged[f'{numerator_col}_test']
        den_e = merged[f'{denominator_col}_test']

# 5. Z-test для кожного рядка
# об'єднанння 4 масиви попарно і передача в z_test_proportion
# результат - список з (p1, p2, diff, z, pval, significant).
        results = [z_test_proportion(nc, dc, ne, de)
            for nc, dc, ne, de in zip(num_c, den_c, num_e, den_e)]

        res_df = pd.DataFrame(results, columns=[
            'conversion_rate_control',
            'conversion_rate_test',
            'metric_change',
            'z_stat',
            'p_value',
            'significant'])

        section_df = pd.DataFrame({
            'test_number': merged['test_number'].values,
            'metric': f'{numerator_col} / {denominator_col}',
            'numerator_event': numerator_col,
            'denominator_event': denominator_col,
            'numerator_control': num_c.values,
            'denominator_control': den_c.values,
            'conversion_rate_control': res_df['conversion_rate_control'].values,
            'numerator_test': num_e.values,
            'denominator_test': den_e.values,
            'conversion_rate_test': res_df['conversion_rate_test'].values,
            'metric_change': res_df['metric_change'].values,
            'z_stat': res_df['z_stat'].values,
            'p_value': res_df['p_value'].values,
            'significant': res_df['significant'].values})

# 6. Збір таблиці з результатом.
        records.append(section_df)

  return pd.concat(records, ignore_index=True)

In [15]:
results_total = cal_signif_total(df)
print(results_total.to_string(index=False))

 test_number                      metric   numerator_event denominator_event  numerator_control  denominator_control  conversion_rate_control  numerator_test  denominator_test  conversion_rate_test  metric_change    z_stat  p_value  significant
           1  add_payment_info / session  add_payment_info           session             1988.0              45362.0                 0.043825          2229.0           45193.0              0.049322       0.005497  3.924884 0.000043         True
           2  add_payment_info / session  add_payment_info           session             2344.0              50637.0                 0.046290          2409.0           50244.0              0.047946       0.001656  1.240994 0.107304        False
           3  add_payment_info / session  add_payment_info           session             3623.0              70047.0                 0.051722          3697.0           70439.0              0.052485       0.000763  0.643172 0.260056        False
           4  add_pa

In [16]:
results_total.to_csv('ab_results_total.csv', index=False)
print('\n✅ Save: ab_results_total.csv')


✅ Save: ab_results_total.csv


In [17]:
files.download('ab_results_total.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# ***✅Аналіз по тесту + девайсу***

In [18]:
def cal_signif_device(data):
  group_cols_device = ['test_number', 'device']
  metric_cols = list({col for col, _ in metric_name } | {'session'})

  agg_device = (data.groupby(group_cols_device + ['variant'])[metric_cols].sum().reset_index())

  control = agg_device[agg_device['variant'] == 'control'].drop(columns='variant')
  test = agg_device[agg_device['variant'] == 'test'].drop(columns='variant')

  merged = control.merge(test, on=group_cols_device, suffixes=('_control', '_test'))

  records_device = []

  for numerator_col, denominator_col in metric_name:
        num_c = merged[f'{numerator_col}_control']
        den_c = merged[f'{denominator_col}_control']
        num_e = merged[f'{numerator_col}_test']
        den_e = merged[f'{denominator_col}_test']

        results_device = [z_test_proportion(nc, dc, ne, de)
            for nc, dc, ne, de in zip(num_c, den_c, num_e, den_e)]

        res_df_device = pd.DataFrame(results_device, columns=[
            'conversion_rate_control',
            'conversion_rate_test',
            'metric_change',
            'z_stat',
            'p_value',
            'significant'])

        section_df_device = pd.DataFrame({
            'test_number': merged['test_number'].values,
            'device': merged['device'].values,
            'metric': f'{numerator_col} / {denominator_col}',
            'numerator_event': numerator_col,
            'denominator_event': denominator_col,
            'numerator_control': num_c.values,
            'denominator_control': den_c.values,
            'conversion_rate_control': res_df_device['conversion_rate_control'].values,
            'numerator_test': num_e.values,
            'denominator_test': den_e.values,
            'conversion_rate_test': res_df_device['conversion_rate_test'].values,
            'metric_change': res_df_device['metric_change'].values,
            'z_stat': res_df_device['z_stat'].values,
            'p_value': res_df_device['p_value'].values,
            'significant': res_df_device['significant'].values})

        records_device.append(section_df_device)

  return pd.concat(records_device, ignore_index=True)

In [19]:
results_device_df = cal_signif_device(df)
print(results_device_df.to_string(index=False))

 test_number  device                      metric   numerator_event denominator_event  numerator_control  denominator_control  conversion_rate_control  numerator_test  denominator_test  conversion_rate_test  metric_change    z_stat  p_value  significant
           1 desktop  add_payment_info / session  add_payment_info           session             1130.0              26467.0                 0.042695          1256.0           26417.0              0.047545       0.004850  2.686998 0.003605         True
           1  mobile  add_payment_info / session  add_payment_info           session              810.0              17896.0                 0.045262           942.0           17767.0              0.053020       0.007758  3.389330 0.000350         True
           1  tablet  add_payment_info / session  add_payment_info           session               48.0                999.0                 0.048048            31.0            1009.0              0.030723      -0.017325 -1.996608 0.977066  

In [20]:
results_device_df.to_csv('ab_results_device.csv', index=False)
print('\n✅ Save: ab_results_device.csv')


✅ Save: ab_results_device.csv


In [21]:
files.download('ab_results_device.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# ***✅Аналіз по тесту + країні***

In [24]:
def cal_signif_country(data):
  group_cols_country = ['test_number', 'country']
  metric_cols = list({col for col, _ in metric_name } | {'session'})

  agg_country = (data.groupby(group_cols_country + ['variant'])[metric_cols].sum().reset_index())

  control = agg_country[agg_country['variant'] == 'control'].drop(columns='variant')
  test = agg_country[agg_country['variant'] == 'test'].drop(columns='variant')

  merged = control.merge(test, on=group_cols_country, suffixes=('_control', '_test'))

  records_country = []

  for numerator_col, denominator_col in metric_name:
        num_c = merged[f'{numerator_col}_control']
        den_c = merged[f'{denominator_col}_control']
        num_e = merged[f'{numerator_col}_test']
        den_e = merged[f'{denominator_col}_test']

        results_country = [z_test_proportion(nc, dc, ne, de)
            for nc, dc, ne, de in zip(num_c, den_c, num_e, den_e)]

        res_df_country = pd.DataFrame(results_country, columns=[
            'conversion_rate_control',
            'conversion_rate_test',
            'metric_change',
            'z_stat',
            'p_value',
            'significant'])

        section_df_country = pd.DataFrame({
            'test_number': merged['test_number'].values,
            'country': merged['country'].values,
            'metric': f'{numerator_col} / {denominator_col}',
            'numerator_event': numerator_col,
            'denominator_event': denominator_col,
            'numerator_control': num_c.values,
            'denominator_control': den_c.values,
            'conversion_rate_control': res_df_country['conversion_rate_control'].values,
            'numerator_test': num_e.values,
            'denominator_test': den_e.values,
            'conversion_rate_test': res_df_country['conversion_rate_test'].values,
            'metric_change': res_df_country['metric_change'].values,
            'z_stat': res_df_country['z_stat'].values,
            'p_value': res_df_country['p_value'].values,
            'significant': res_df_country['significant'].values})

        records_country.append(section_df_country)

  return pd.concat(records_country, ignore_index=True)

In [25]:
results_country_df = cal_signif_country(df)
print(results_country_df.to_string(index=False))

/tmp/ipykernel_9549/900433190.py:6: RuntimeWarning: invalid value encountered in sqrt
  se = np.sqrt(p_pool * (1 - p_pool) * (1 / den_control + 1 / den_test))


 test_number              country                      metric   numerator_event denominator_event  numerator_control  denominator_control  conversion_rate_control  numerator_test  denominator_test  conversion_rate_test  metric_change    z_stat      p_value  significant
           1            (not set)  add_payment_info / session  add_payment_info           session               16.0                369.0                 0.043360            19.0             373.0              0.050938       0.007578  0.486827 3.131904e-01        False
           1              Albania  add_payment_info / session  add_payment_info           session                1.0                  9.0                 0.111111             0.0              16.0              0.000000      -0.111111 -1.360828 9.132159e-01        False
           1              Algeria  add_payment_info / session  add_payment_info           session                2.0                 29.0                 0.068966             3.0            

In [ ]:
results_country_df.to_csv('ab_results_country.csv', index=False)
print('\n✅ Save: ab_results_country.csv')

In [ ]:
files.download('ab_results_country.csv')